# CNN-MCL → BI-LSTM → FWA → RandomForest on NSL-KDD

Two-phase pipeline: (1) train the DL extractor with a temporary softmax head, (2) fit a RandomForest on the extracted 4H features. Paper reference: Hashmi, Barukab & Hamza Osman, PLOS ONE 19(5), 2024.

In [13]:
!pip install speedtest-cli --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [15]:
!pip install torch==2.12.0+cu128 torchvision==0.27.0+cu128 \
    --extra-index-url https://download.pytorch.org/whl/cu128
!pip install numpy==2.4.6 pandas==3.0.3 scikit-learn==1.8.0 scipy==1.17.1 statsmodels==0.14.6
!pip install matplotlib==3.10.9 seaborn==0.13.2 plotly==6.7.0

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [8]:
!pip list

Package                 Version
----------------------- ---------------
attrs                   23.2.0
Automat                 22.10.0
Babel                   2.10.3
bcrypt                  3.2.2
blinker                 1.7.0
certifi                 2023.11.17
chardet                 5.2.0
click                   8.1.6
cloud-init              25.2
colorama                0.4.6
command-not-found       0.3
configobj               5.0.8
constantly              23.10.4
contourpy               1.3.3
cryptography            41.0.7
cycler                  0.12.1
dbus-python             1.3.2
distro                  1.9.0
distro-info             1.7+build1
fonttools               4.63.0
httplib2                0.20.4
hyperlink               21.0.0
idna                    3.6
incremental             22.10.0
Jinja2                  3.1.2
joblib                  1.5.3
jsonpatch               1.32
jsonpointer             2.0
jsonschema              4.10.3
kiwisolver              1.5.0
launchpadlib

In [10]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import torch


from nids_dl import RFConfig, TrainConfig, evaluate, extract_features, fit_rf, train_extractor
from nids_dl.data import load_processed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
  print(f"Using {torch.cuda.device_count()} GPU(s):")
  for i in range(torch.cuda.device_count()):
      print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}")
else:
  print("No GPU found, using CPU")


ModuleNotFoundError: No module named 'torch'

In [3]:
DEVICE

'cpu'

In [1]:
!nvidia-smi

Sun May 24 01:40:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:02:00.0  On |                  N/A |
|  0%   47C    P8             36W /  600W |     761MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Load preprocessed NSL-KDD

In [5]:
train = load_processed(ROOT / "data" / "processed" / "train.pt")
test = load_processed(ROOT / "data" / "processed" / "test.pt")

X_tr, y_tr = train["X"], train["y_bin"]
X_te, y_te = test["X"], test["y_bin"]
X_tr.shape, X_te.shape, int(y_tr.max().item()) + 1

(torch.Size([125973, 120]), torch.Size([22544, 120]), 2)

## 2. Phase 1 — train the DL feature extractor

Cross-entropy on the temporary softmax head, Adam, with the MCL prediction-error-filter constraint re-applied after every step.

In [6]:
cfg = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="binary", log_every=0, seed=0)
extractor, history = train_extractor(X_tr, y_tr, cfg, X_val=X_te, y_val=y_te)
history[-1]

{'epoch': 9,
 'train_loss': 0.06449754264443129,
 'train_acc': 0.9767410476848214,
 'val_loss': 0.683620837751731,
 'val_acc': 0.7924503193754435}

In [7]:
import pandas as pd

pd.DataFrame(history)

,epoch,train_loss,train_acc,val_loss,val_acc
0,0,0.318324,0.854596,0.405727,0.807887
1,1,0.164045,0.941218,0.602397,0.743923
2,2,0.139696,0.955260,0.930190,0.734475
3,3,0.104555,0.968160,0.623272,0.762864
4,4,0.088063,0.971232,0.771187,0.742104
5,5,0.080357,0.972510,0.746913,0.745342
6,6,0.072323,0.974352,0.766912,0.757053
7,7,0.068774,0.975098,0.601486,0.786817
8,8,0.063666,0.976614,0.611984,0.796309
9,9,0.064498,0.976741,0.683621,0.792450


## 3. Extract features and fit RandomForest

In [8]:
Fe_tr = extract_features(extractor, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te = extract_features(extractor, X_te, batch_size=512, device=DEVICE).numpy()
Fe_tr.shape, Fe_te.shape

((125973, 256), (22544, 256))

In [9]:
clf = fit_rf(Fe_tr, y_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_train = evaluate(clf, Fe_tr, y_tr.numpy())
metrics_test = evaluate(clf, Fe_te, y_te.numpy())
{
    "train": {k: metrics_train[k] for k in ("accuracy", "precision", "recall", "f1")},
    "test": {k: metrics_test[k] for k in ("accuracy", "precision", "recall", "f1")},
}

{'train': {'accuracy': 0.999944432537131,
  'precision': 0.9999488307834007,
  'recall': 0.9999317755415317,
  'f1': 0.9999403030897415},
 'test': {'accuracy': 0.7791430092264017,
  'precision': 0.9266623207301173,
  'recall': 0.6646146653159822,
  'f1': 0.774061805145891}}

In [10]:
print(metrics_test["report"])
metrics_test["confusion_matrix"]

              precision    recall  f1-score   support

           0       0.68      0.93      0.78      9711
           1       0.93      0.66      0.77     12833

    accuracy                           0.78     22544
   macro avg       0.80      0.80      0.78     22544
weighted avg       0.82      0.78      0.78     22544



array([[9036,  675],
       [4304, 8529]])

## 4. Multi-class variant (5 classes: Normal/DoS/Probe/R2L/U2R)

In [11]:
ym_tr, ym_te = train["y_mul"], test["y_mul"]
cfg_m = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="multi", seed=0)
extractor_m, history_m = train_extractor(X_tr, ym_tr, cfg_m, X_val=X_te, y_val=ym_te)
Fe_tr_m = extract_features(extractor_m, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te_m = extract_features(extractor_m, X_te, batch_size=512, device=DEVICE).numpy()
clf_m = fit_rf(Fe_tr_m, ym_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_te_m = evaluate(clf_m, Fe_te_m, ym_te.numpy())
print(metrics_te_m["report"])
{k: metrics_te_m[k] for k in ("accuracy", "precision", "recall", "f1")}

              precision    recall  f1-score   support

           0       0.65      0.98      0.78      9711
           1       0.97      0.77      0.86      7460
           2       0.83      0.60      0.70      2421
           3       0.96      0.03      0.06      2885
           4       0.00      0.00      0.00        67

    accuracy                           0.75     22544
   macro avg       0.68      0.48      0.48     22544
weighted avg       0.81      0.75      0.70     22544



{'accuracy': 0.7470723917672107,
 'precision': 0.6798153446608264,
 'recall': 0.47748296099652504,
 'f1': 0.4796168070336827}